# IoT IDS — Training Notebook

Trains the **Stage 1 (unsupervised anomaly + binary gate)** and **Stage 2 (multiclass attack classifier)**
models described in `docs/IoT-IDS-Project-Plan.md`.

This notebook is intentionally dependency-light (scikit-learn only, no torch/lightgbm/torch-geometric) so it
runs anywhere with zero extra setup. The plan's full architecture also specifies a GraphSAGE topology model and
an LSTM sequence model as later upgrades — this notebook builds the working V1 slice: signature-free anomaly
detection + a calibrated binary/multiclass ensemble, which is already a real, working two-stage IDS.

**What this notebook produces**, saved into `../backend/app/models/`:
- `anomaly_model.pkl` — IsolationForest, trained on benign traffic only
- `gate_model.pkl` — binary attack-vs-benign classifier, tuned to a hard FPR ceiling
- `multiclass_model.pkl` — attack-type classifier (port_scan / ddos / brute_force / mirai_beacon)
- `gate_threshold.json` — the FPR-constrained decision threshold selected below
- `feature_scaler.pkl` — StandardScaler fit on training data, used identically at inference

Run all cells top to bottom.

In [ ]:
import sys, json, time
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.append(".")  # so `features.py` and `synthetic_traffic.py` in this folder import cleanly

from features import FEATURE_COLUMNS, ATTACK_LABELS, LABEL_TO_ID, DeviceState, build_feature_vector
from synthetic_traffic import generate_dataset, make_device_pool

MODEL_OUT_DIR = Path("../backend/app/models")
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Models will be saved to:", MODEL_OUT_DIR.resolve())

: 

## 1. Generate (or load) training data

By default this generates a synthetic dataset that mirrors real IoT traffic shape (see
`synthetic_traffic.py` for the exact per-class logic). **To train on a real public dataset instead**
(CICIoT2023 / TON_IoT / Bot-IoT — see `docs/IoT-IDS-Project-Plan.md` §8 for links), replace the call
below with `synthetic_traffic.load_real_dataset('path/to.csv')` after implementing the column mapping
for that dataset.

In [ ]:
RAW_FLOWS = generate_dataset(n_devices=10, n_flows=30000, attack_ratio=0.15, seed=7)
raw_df = pd.DataFrame(RAW_FLOWS)
raw_df = raw_df.sort_values("ts").reset_index(drop=True)   # enforce chronological order
print(raw_df.shape)
raw_df["label"].value_counts()

## 2. Feature engineering (leakage-safe)

Uses the exact same `features.py` module the backend imports at inference time — this is what keeps
training and serving in sync. Per-device rolling state (`DeviceState`) is rebuilt here in chronological
order, exactly as the backend will build it live, flow by flow.

In [ ]:
device_states = {}   # device_id -> DeviceState, rebuilt in chronological order (no shuffling!)
feature_rows = []
labels = []

for _, flow in raw_df.iterrows():
    flow = flow.to_dict()
    dev_id = flow["device_id"]
    if dev_id not in device_states:
        device_states[dev_id] = DeviceState()
    vec = build_feature_vector(flow, device_states[dev_id])
    feature_rows.append(vec)
    labels.append(flow["label"])

X = np.array(feature_rows, dtype=float)
y_label = np.array(labels)
y_binary = (y_label != "benign").astype(int)
y_multi = np.array([LABEL_TO_ID[l] for l in y_label])

print("Feature matrix:", X.shape)
print("Feature columns:", FEATURE_COLUMNS)

## 3. Chronological split

Never a random row split for an IDS — see plan §5/§9.2. We split by TIME: earliest 70% for training,
next 15% for validation (used for threshold selection + early stopping style checks), last 15% held out
as the final test set.

In [ ]:
n = len(X)
i_train = int(n * 0.70)
i_val = int(n * 0.85)

X_train, y_bin_train, y_multi_train, y_label_train = X[:i_train], y_binary[:i_train], y_multi[:i_train], y_label[:i_train]
X_val,   y_bin_val,   y_multi_val,   y_label_val   = X[i_train:i_val], y_binary[i_train:i_val], y_multi[i_train:i_val], y_label[i_train:i_val]
X_test,  y_bin_test,  y_multi_test,  y_label_test  = X[i_val:], y_binary[i_val:], y_multi[i_val:], y_label[i_val:]

print("train/val/test sizes:", len(X_train), len(X_val), len(X_test))

## 4. Scale features

Fit ONLY on training data, applied identically to val/test/live inference (the saved scaler ships to
the backend so this is never re-fit on live traffic).

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

joblib.dump(scaler, MODEL_OUT_DIR / "feature_scaler.pkl")
print("scaler saved")

## 5. Stage 1a — Unsupervised anomaly model

Trained **only on benign traffic** — needs no attack labels at all. This is the component with a real
chance of flagging an attack type that was never in any labeled dataset (plan §9.4).

In [ ]:
from sklearn.ensemble import IsolationForest

benign_mask_train = (y_label_train == "benign")
anomaly_model = IsolationForest(
    n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1
)
anomaly_model.fit(X_train_s[benign_mask_train])

# score_samples: higher = more normal. We flip sign so higher = more anomalous, easier to reason about.
val_anomaly_score = -anomaly_model.score_samples(X_val_s)
joblib.dump(anomaly_model, MODEL_OUT_DIR / "anomaly_model.pkl")
print("anomaly model trained on", benign_mask_train.sum(), "benign flows")

## 6. Stage 1b — Binary gate (attack vs benign), tuned to a hard FPR ceiling

This is the ONLY component tuned directly against the false-positive budget (plan §9.5/§9.7). We
deliberately use conservative hyperparameters (`min_samples_leaf` high) since this gate sees 100% of
traffic and every false positive here becomes an analyst-facing alert.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, classification_report

gate_model = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=25,
    class_weight="balanced", random_state=42, n_jobs=-1
)
gate_model.fit(X_train_s, y_bin_train)

val_gate_score = gate_model.predict_proba(X_val_s)[:, 1]
joblib.dump(gate_model, MODEL_OUT_DIR / "gate_model.pkl")
print(classification_report(y_bin_val, gate_model.predict(X_val_s), target_names=["benign", "attack"]))

In [ ]:
def select_threshold_for_fpr(y_true_binary, y_scores, max_fpr=0.02):
    """Pick the highest-recall threshold that keeps FPR under the ops budget —
    NOT the threshold that maximizes F1 (see plan §9.7 for why)."""
    fpr, tpr, thr = roc_curve(y_true_binary, y_scores)
    valid = fpr <= max_fpr
    if not valid.any():
        raise ValueError(f"No threshold achieves FPR <= {max_fpr}; model needs more work")
    best_idx = np.argmax(tpr[valid])
    return float(thr[valid][best_idx]), float(fpr[valid][best_idx]), float(tpr[valid][best_idx])

GATE_THRESHOLD, achieved_fpr, achieved_tpr = select_threshold_for_fpr(y_bin_val, val_gate_score, max_fpr=0.02)
print(f"Selected gate threshold={GATE_THRESHOLD:.4f} -> validation FPR={achieved_fpr:.4f}, recall={achieved_tpr:.4f}")

with open(MODEL_OUT_DIR / "gate_threshold.json", "w") as f:
    json.dump({"gate_threshold": GATE_THRESHOLD, "target_max_fpr": 0.02,
                "achieved_val_fpr": achieved_fpr, "achieved_val_recall": achieved_tpr}, f, indent=2)

## 7. Stage 2 — Multiclass attack-type classifier

Runs (in the backend) only on flows the Stage-1 gate already flagged — so it only needs to answer
"what kind of attack is this", a lower-stakes question than "is this an attack at all".

In [ ]:
multiclass_model = RandomForestClassifier(
    n_estimators=400, max_depth=None, min_samples_leaf=5,
    class_weight="balanced", random_state=42, n_jobs=-1
)
multiclass_model.fit(X_train_s, y_multi_train)
joblib.dump(multiclass_model, MODEL_OUT_DIR / "multiclass_model.pkl")

val_pred_multi = multiclass_model.predict(X_val_s)
print(classification_report(y_multi_val, val_pred_multi, target_names=ATTACK_LABELS))

## 8. Final held-out test evaluation

Report **per-class** metrics — never a single blended accuracy number (plan §5/§17). This is the
number that matters, since it was never touched during threshold selection or hyperparameter choices.

In [ ]:
from sklearn.metrics import confusion_matrix

test_gate_score = gate_model.predict_proba(X_test_s)[:, 1]
test_gate_pred = (test_gate_score >= GATE_THRESHOLD).astype(int)
print("=== Stage 1 gate — held-out test ===")
print(classification_report(y_bin_test, test_gate_pred, target_names=["benign", "attack"]))

test_multi_pred = multiclass_model.predict(X_test_s)
print("=== Stage 2 multiclass — held-out test (on ALL test flows, for reporting) ===")
print(classification_report(y_multi_test, test_multi_pred, target_names=ATTACK_LABELS))

print("Confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(y_multi_test, test_multi_pred),
                    index=ATTACK_LABELS, columns=ATTACK_LABELS))

## 9. Hard-negative check (should NOT alert)

Per plan §10 — explicitly test traffic that looks unusual but is legitimately benign (e.g. a device
suddenly downloading a large firmware update). If the gate fires here, the FPR ceiling picked above is
too aggressive for this feature set and needs revisiting before going anywhere near real traffic.

In [ ]:
firmware_update_flow = {
    "ts": time.time(), "device_id": "camera-demo", "src_ip": "192.168.1.10",
    "dst_ip": "203.0.113.5", "src_port": 51000, "dst_port": 443, "proto": "tcp",
    "duration": 45.0, "orig_bytes": 2_000_000, "resp_bytes": 40_000_000,
    "orig_pkts": 3000, "resp_pkts": 28000, "conn_state": "SF",
}
hard_neg_state = DeviceState()
# warm up baseline with a few normal camera flows first so this isn't unfairly judged as brand-new
for _ in range(10):
    build_feature_vector(
        {**firmware_update_flow, "orig_bytes": 180_000, "resp_bytes": 120_000,
         "orig_pkts": 300, "resp_pkts": 250, "duration": 8.0},
        hard_neg_state,
    )
vec = np.array([build_feature_vector(firmware_update_flow, hard_neg_state)])
vec_s = scaler.transform(vec)
score = gate_model.predict_proba(vec_s)[0, 1]
print(f"Firmware-update hard-negative gate score: {score:.4f} (threshold={GATE_THRESHOLD:.4f}) "
      f"-> {'ALERTS (investigate!)' if score >= GATE_THRESHOLD else 'correctly stays quiet'}")

## 10. Save feature column order

The backend needs to know the exact feature order this model expects — save it alongside the models
so a future feature-set change fails loudly instead of silently misaligning columns.

In [ ]:
with open(MODEL_OUT_DIR / "feature_columns.json", "w") as f:
    json.dump({"feature_columns": FEATURE_COLUMNS, "attack_labels": ATTACK_LABELS}, f, indent=2)

print("All artifacts saved to", MODEL_OUT_DIR.resolve())
print(sorted(p.name for p in MODEL_OUT_DIR.glob("*")))

## Next steps (see `docs/IoT-IDS-Project-Plan.md` for the full roadmap)

- Swap synthetic data for a real dataset via `synthetic_traffic.load_real_dataset(...)`, then also add
  the **cross-dataset generalization test** from plan §9.2/§17 (train on dataset A, evaluate on B with
  zero fine-tuning) before trusting these numbers.
- Add the GraphSAGE device-topology model and LSTM per-device sequence model from plan §9.6 as
  additional Stage-2 ensemble members once this working slice is validated end-to-end.
- Add SHAP-based per-alert explanations (plan §10) — `multiclass_model.feature_importances_` is used as
  a lightweight stand-in for this in the backend for now.